# 배송지 변경 실습 · 도구 오류와 체크포인트 분기

**먼저 `USE = "fake"`로 위에서 아래로 실행하세요.** Solar 준비 셀은 건너뛰어도 됩니다. 가짜 모델은 GPU나 모델 API를 쓰지 않습니다.

고객이 출고 전 주문 O001의 배송지를 바꾸려 합니다. 배송 조회는 첫 시도에 시간 초과가 납니다. 조회를 다시 해 보고, 배송지 변경은 고객이 승인한 뒤 실행합니다. 그다음 조회 직후로 돌아가 같은 요청을 재생하거나 다른 배송지로 분기해 봅니다.

용어는 [앞 실습](02_대화_기억과_사람의_승인.ipynb)에 나온 것에서 조금만 더 보탭니다. 메시지, 상태, 체크포인터, 미들웨어, HITL은 앞 실습을 참고하세요.

| 새 용어 | 여기서 뜻하는 것 |
|---|---|
| 자동 재시도 | 미들웨어가 실패한 도구 요청을 다시 실행하는 일입니다. |
| 재생(replay) | 저장한 체크포인트를 골라 남은 단계부터 다시 실행하는 일입니다. |
| 분기(fork) | 과거 체크포인트의 상태를 수정해 새 경로로 이어 가는 일입니다. |
| 부수 효과(side effect) | 도구가 대화 밖의 데이터를 바꾸는 일입니다. 배송지 변경과 실행 기록이 예입니다. |

셀 번호 대신 제목과 짧은 셀 ID를 함께 적었습니다. 결과가 섞이면 커널을 재시작하고 위에서부터 실행하세요.

## 준비 · 프로젝트와 출력 도우미

앞 실습처럼 프로젝트 경로를 맞춥니다. 「환경과 도우미」 셀(`setup`)은 현재 커널의 Python, LangChain, LangGraph와 모델 연결 패키지 버전을 출력합니다.

`show()`는 메시지를 한 줄씩 보여 줍니다. 모델이 요청한 도구는 이름·입력·호출 ID까지 보입니다. 도구 결과의 `call` 값은 응답 대상인 호출 ID입니다. `status`는 도구 응답에 붙는 표식입니다. 배송지 변경이 완료됐는지는 응답 내용의 `ok`로 확인하세요.

`thread_config()`는 새 대화 이름표를 만듭니다. 이름표에 붙는 값과 체크포인트 ID는 실행할 때마다 달라집니다. 가짜 모델의 도구 선택과 배송지 변경 순서는 같습니다.

In [ ]:
import json
import os
import re
import subprocess
import sys
from importlib.metadata import version
from pathlib import Path
from typing import Annotated
from uuid import uuid4

REPO = Path("/home/home/projects/shopagent")
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware, ToolRetryMiddleware
from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

print("Python:", sys.version.split()[0])
for package in ("langchain", "langchain-core", "langgraph", "langchain-openai", "ipykernel"):
    print(f"{package}: {version(package)}")


def show(messages):
    for message in messages:
        line = f"{message.type}: {message.content}"
        if isinstance(message, AIMessage) and message.tool_calls:
            line += " | tool_calls=" + json.dumps(message.tool_calls, ensure_ascii=False)
        if isinstance(message, ToolMessage):
            line += f" | name={message.name} status={message.status} call={message.tool_call_id}"
        print(line.replace("\n", " ↵ "))


def thread_config(label):
    return {"configurable": {"thread_id": f"{label}-{uuid4().hex[:8]}"}}

### Gemma · shopagent와 같은 로컬 연결

「Gemma 준비」 셀(`gemma`)은 `LocalLLM(port=18380).client`를 만든 뒤 이 컴퓨터의 `/health`만 최대 2초 동안 확인합니다. 준비 메시지는 연결 객체를 만들었다는 뜻입니다. 다음 줄의 HTTP 상태 코드로 서버의 응답을 확인하세요. 연결 오류가 나면 서버 주소에 응답이 없다는 안내를 띄우고 계속합니다.

Gemma로 실습할 때는 별도 터미널에서 다음 명령으로 서버를 실행할 수 있습니다. 종료하려면 그 터미널에서 `Ctrl+C`를 누르세요. 가짜 모델에는 서버가 필요 없습니다.

~~~bash
/home/home/llama.cpp/build/bin/llama-server -m /home/home/models/gemma-4-12b-it-Q4_K_M.gguf --jinja -ngl 99 -ctk q8_0 -ctv q8_0 -fa on -np 1 -c 16384 --host 127.0.0.1 --port 18380 --n-cpu-moe 0 --chat-template-kwargs '{"enable_thinking": false}' --reasoning-budget 0
~~~

In [ ]:
import httpx
from shop.agents import LocalLLM

gemma_llm = LocalLLM(port=18380).client
print("Gemma 연결 객체 준비 완료. 모델에 질문은 보내지 않았습니다.")
try:
    with httpx.Client(trust_env=False) as health_client:
        health_response = health_client.get("http://127.0.0.1:18380/health", timeout=2)
    print(f"llama-server 응답 상태 코드: {health_response.status_code}")
except httpx.HTTPError:
    print("llama-server에 연결할 수 없습니다. Gemma를 쓸 때 위의 터미널 명령을 실행하세요.")

### Solar · shopagent와 같은 Upstage 설정

**Solar에 실제로 질문하면 비용이 듭니다.** 가짜 모델을 쓸 때는 「Solar 준비」 셀(`solar`)을 건너뛰세요.

앞 실습과 같은 방식으로 bash의 점 명령으로 키 파일을 불러옵니다. 셀은 키를 프로세스 환경에 넣고 키가 준비됐는지만 `True` 또는 `False`로 출력합니다. `False`일 때는 가짜 모델을 선택하라는 안내가 이어집니다. 키, 환경 변수 목록, 셸의 출력과 오류는 표시하지 않습니다.

In [ ]:
from shop.servers import CONFIGS


def load_solar_key():
    try:
        result = subprocess.run(
            ["bash", "-c", 'set -a; . ./upstage.env; set +a; printf %s "$UPSTAGE_API_KEY"'],
            cwd=REPO, capture_output=True, text=True, timeout=5,
        )
        if result.stdout.strip():
            os.environ["UPSTAGE_API_KEY"] = result.stdout.strip()
    except (OSError, subprocess.TimeoutExpired):
        pass
    return bool(os.environ.get("UPSTAGE_API_KEY", "").strip())


solar_loaded = load_solar_key()
print("Upstage 키 준비 여부:", solar_loaded)
solar_llm = LocalLLM(config=CONFIGS["solar-pro4"]).client if solar_loaded else None
if solar_llm is None:
    print("키가 없어 Solar를 준비하지 못했습니다. USE = 'fake'로 진행하세요.")

### 가짜 모델 · 오류를 보고 한 번 더 요청하기

`DeliveryChatModel`은 `BaseChatModel`의 틀을 씁니다. `bind_tools()`는 이 실습의 두 도구를 전달받고 자기 자신을 돌려줍니다. 대화는 모델 객체에 저장하지 않고 매번 전달받은 메시지에서 현재 고객 요청과 도구 결과를 읽습니다.

O001 같은 주문 번호를 찾으면 먼저 조회를 요청합니다. 조회 오류를 한 번 받으면 새 호출 ID를 만들어 한 번 더 요청합니다. 두 번째 오류를 받으면 변경할 수 없었다고 안내합니다. 조회에 성공하고 고객이 ‘새 배송지:’ 뒤에 주소를 적었다면 변경 도구를 요청합니다. 도구 결과를 받은 뒤에 완료 여부를 답합니다.

「가짜 모델 준비」 셀(`fake`)의 한 줄은 규칙을 담은 객체가 만들어졌다는 안내입니다.

In [ ]:
class DeliveryChatModel(BaseChatModel):
    @property
    def _llm_type(self):
        return "delivery-scripted"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        return self

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        reply = self._reply(messages)
        return ChatResult(generations=[ChatGeneration(message=reply)])

    def _reply(self, messages):
        customer_index = next(
            (i for i in range(len(messages) - 1, -1, -1) if isinstance(messages[i], HumanMessage)),
            None,
        )
        if customer_index is None:
            return AIMessage(content="주문 번호를 알려 주세요.")
        customer_text = str(messages[customer_index].content)
        order_match = re.search(r"\bO\d+\b", customer_text, flags=re.ASCII)
        if order_match is None:
            return AIMessage(content="O001과 같은 주문 번호를 알려 주세요.")

        order_id = order_match.group()
        new_address = customer_text.partition("새 배송지:")[2].strip()
        turn = messages[customer_index + 1:]
        latest = messages[-1]
        call_number = 1 + sum(len(m.tool_calls) for m in messages if isinstance(m, AIMessage))

        if not isinstance(latest, ToolMessage):
            return self._call("check_delivery_window", {"order_id": order_id}, call_number)

        if latest.name == "check_delivery_window":
            if latest.status == "error":
                failures = sum(
                    isinstance(m, ToolMessage) and m.name == latest.name and m.status == "error"
                    for m in turn
                )
                if failures == 1:
                    return self._call("check_delivery_window", {"order_id": order_id}, call_number)
                return AIMessage(content="배송 조회가 두 번 실패했어요. 배송지는 변경하지 않았어요.")

            result = json.loads(latest.content)
            if not result["can_change"]:
                return AIMessage(content=f"배송지를 변경할 수 없어요. {result['reason']}")
            if not new_address:
                return AIMessage(content="아직 출고 전이에요. 새 배송지를 알려 주세요.")
            return self._call(
                "change_shipping_address", {"order_id": order_id, "address": new_address}, call_number
            )

        if latest.status == "error":
            return AIMessage(content="배송지를 변경하지 않았어요.")
        result = json.loads(latest.content)
        if not result["ok"]:
            return AIMessage(content=f"배송지를 변경하지 못했어요. {result['reason']}")
        return AIMessage(content=f"배송지를 변경했어요. 새 주소는 {result['address']}입니다.")

    @staticmethod
    def _call(name, args, call_number):
        return AIMessage(content="", tool_calls=[{
            "name": name, "args": args, "id": f"call-{call_number}", "type": "tool_call",
        }])


fake_llm = DeliveryChatModel()
print("가짜 모델 준비 완료. GPU와 모델 API 없이 실행합니다.")

### 모델 선택

「모델 선택」 셀(`choice`)은 이후 셀에서 쓸 `llm`을 고르고 이름을 출력합니다. 기본값은 `fake`입니다. `gemma`는 위 준비 셀의 서버 응답을 확인하고, `solar`는 키 로드 결과를 확인하세요. 준비되지 않은 모델을 고르면 안내와 함께 선택 셀에서 멈춥니다. 모델을 바꿨다면 이 셀부터 다시 실행하세요.

아래 예측은 가짜 모델 기준입니다. 실제 모델은 도구 선택이나 호출 순서가 달라질 수 있습니다.

In [ ]:
USE = "fake"  # "fake" | "gemma" | "solar"
models = {"fake": fake_llm, "gemma": globals().get("gemma_llm"), "solar": globals().get("solar_llm")}
assert USE in models, "USE는 fake, gemma, solar 중 하나여야 합니다."
llm = models[USE]
assert llm is not None, "해당 모델을 준비하거나 USE = 'fake'로 바꾸세요."
print("선택한 모델:", USE)

## 1. 한 번 실패하는 배송 조회

「주문과 도구」 셀(`toy-tools`)은 주문, 남은 조회 오류 횟수, 실제 도구 실행 기록을 작은 딕셔너리와 목록으로 만듭니다. 이 자료는 배송 API와 주문 데이터베이스 역할을 합니다. 두 예시가 서로 영향을 주지 않도록 각각 새 장난감 데이터를 만듭니다.

`check_delivery_window`는 첫 시도에 오류 횟수를 하나 줄이고 `CourierTimeout`을 일으킵니다. 그 다음부터는 출고 상태를 읽어 변경 가능 여부를 돌려줍니다. 조회는 새 배송지와 무관합니다. `change_shipping_address`는 출고 전인지 다시 확인하고 주소를 바꿉니다. 실행 기록에는 변경 전과 후의 주소를 남깁니다.

이 자료는 에이전트 상태 바깥에 둡니다. 덕분에 나중에 과거 체크포인트를 열었을 때 메시지는 과거의 것인데 주문 주소는 지금 값인 모습을 볼 수 있습니다.

In [ ]:
class CourierTimeout(TimeoutError):
    """장난감 배송 조회가 시간 내에 끝나지 않았습니다."""


def make_toy_shop(failures=1):
    return {
        "orders": {"O001": {"status": "상품준비중", "address": "서울시 성동구 새길 10"}},
        "courier": {"failures_left": failures},
        "tool_runs": [],
    }


def make_delivery_tools(shop):
    orders = shop["orders"]
    courier = shop["courier"]
    tool_runs = shop["tool_runs"]

    @tool
    def check_delivery_window(order_id: Annotated[str, "출고 상태를 조회할 주문 번호"]) -> dict:
        """배송지를 변경할 수 있는지 조회합니다. 주소를 저장하지 않습니다."""
        if courier["failures_left"] > 0:
            courier["failures_left"] -= 1
            tool_runs.append(("조회", order_id, "시간 초과"))
            raise CourierTimeout("배송 조회 시간 초과")

        order = orders.get(order_id)
        tool_runs.append(("조회", order_id, "응답"))
        return {
            "order_id": order_id,
            "can_change": bool(order and order["status"] == "상품준비중"),
            "reason": f"주문 상태는 {order['status']}입니다." if order else "주문을 찾을 수 없습니다.",
        }

    @tool
    def change_shipping_address(
        order_id: Annotated[str, "배송지를 바꿀 주문 번호"],
        address: Annotated[str, "고객이 요청한 새 배송지 전체"],
    ) -> dict:
        """출고 전 주문의 배송지를 변경합니다. 실행 직전에 고객 확인을 받습니다."""
        order = orders.get(order_id)
        if order is None or order["status"] != "상품준비중":
            tool_runs.append(("변경", order_id, "변경 불가"))
            return {"ok": False, "reason": "변경할 수 있는 주문이 아닙니다."}

        previous_address = order["address"]
        order["address"] = address
        tool_runs.append(("변경", order_id, previous_address, address))
        return {"ok": True, "order_id": order_id, "address": address}

    return [check_delivery_window, change_shipping_address]

### 어느 단계에서 다시 시도할까

「에이전트와 승인 도우미」 셀(`delivery-agent`)은 `create_agent`로 ‘모델 → 도구 → 모델’ 흐름을 만듭니다. `InMemorySaver`는 단계별 상태를 이 커널에 저장합니다.

설치된 `ToolRetryMiddleware`의 `max_retries`는 첫 실행 **이후**에 허용할 재시도 횟수입니다. `tools`와 `retry_on`으로 배송 조회의 `CourierTimeout`만 고릅니다. 0이면 첫 오류에서 바로 `on_failure`를 부릅니다. 예외를 받은 이 함수는 모델에게 보낼 짧은 안내를 돌려줍니다. 1이면 같은 요청으로 한 번 더 실행하고, 그래도 실패하면 `on_failure`를 부릅니다. `on_failure`의 안내는 원래 호출 ID를 가진 `ToolMessage(status="error")`로 모델에게 갑니다. 실습에서는 기다리는 시간을 0으로 둡니다.

`HumanInTheLoopMiddleware`는 배송지 변경 직전에만 멈춥니다. 승인 도우미는 최신 상태의 중단 ID에 `approve`를 연결해 재개합니다. 상태 출력에서 ‘승인 대기’는 실제 중단의 확인 문구이며, 빈 목록이면 대기 중인 승인이 없습니다. ‘현재 배송지’와 ‘실제 도구 실행’은 그래프 밖 데이터에서 읽습니다.

In [ ]:
delivery_prompt = (
    "배송지 변경을 돕는 고객 상담원입니다. 먼저 check_delivery_window를 호출하세요. "
    "시간 초과 결과를 받으면 모델에서 조회를 한 번만 다시 요청하세요. 두 번째 오류 뒤에는 고객에게 설명하세요. "
    "can_change가 참이고 고객이 새 주소를 주었으면 change_shipping_address를 정확히 한 번 요청하세요. "
    "도구 실행 전에 프로그램이 승인을 받습니다. 변경 결과의 ok가 참일 때만 완료를 안내하세요. "
    "한 응답에 도구는 한 개만 요청하세요."
)


def format_courier_error(_error):
    return "배송 조회 시간 초과. 다시 조회할 수 있습니다."


def describe_address_change(call, state, runtime):
    args = call["args"]
    return f"주문 {args['order_id']}의 배송지를 바꿀까요? 새 주소는 {args['address']}입니다."


def make_delivery_agent(shop, *, tool_retries=0):
    retry_lookup = ToolRetryMiddleware(
        max_retries=tool_retries,
        tools=["check_delivery_window"],
        retry_on=(CourierTimeout,),
        on_failure=format_courier_error,
        initial_delay=0,
        jitter=False,
    )
    approve_change = HumanInTheLoopMiddleware(
        interrupt_mode="per_call",
        interrupt_on={"change_shipping_address": {
            "allowed_decisions": ["approve", "reject"],
            "description": describe_address_change,
        }},
    )
    return create_agent(
        llm, make_delivery_tools(shop), system_prompt=delivery_prompt,
        middleware=[approve_change, retry_lookup], checkpointer=InMemorySaver(),
    )


def show_delivery(agent, config, result, shop):
    show(result["messages"])
    state = agent.get_state(config)
    print("승인 대기:", [pending.value["description"] for pending in state.interrupts])
    print("현재 배송지:", shop["orders"]["O001"]["address"])
    print("실제 도구 실행:", shop["tool_runs"])


def approve_delivery(agent, config):
    pending = agent.get_state(config).interrupts
    if not pending:
        print("승인할 배송지 변경이 없어 재개를 건너뜁니다.")
        return None
    return agent.invoke(
        Command(resume={item.id: {"type": "approve"} for item in pending}), config
    )

### 모델이 오류를 받은 뒤 직접 재시도

「모델 재시도」 셀(`model-retry`)은 미들웨어의 재시도를 0회로 둡니다. 첫 조회의 `status=error`는 `call-1` 응답입니다. 가짜 모델은 그 내용을 받아 새 ID인 `call-2`로 조회를 요청합니다. 조회 성공 뒤에는 `call-3`으로 배송지 변경을 요청합니다.

‘승인 대기’에 왕십리로 20을 묻는 문구가 보입니다. 아직 주소는 새길 10이고 실제 실행 목록에는 조회 두 건만 있습니다.

In [ ]:
customer = HumanMessage(
    id="delivery-request",
    content="주문 O001의 배송지를 바꿔 주세요. 새 배송지: 서울시 성동구 왕십리로 20",
)

original_shop = make_toy_shop()
original_agent = make_delivery_agent(original_shop, tool_retries=0)
original_config = thread_config("delivery-model-retry")
original_pause = original_agent.invoke({"messages": [customer]}, original_config)
show_delivery(original_agent, original_config, original_pause, original_shop)

### 승인을 보내고 현재 배송지 확인

「원래 요청 승인」 셀(`approve-original`)은 위의 승인 대기 상태에서 재개합니다. 가짜 모델에서 도구 응답이 추가되고 ‘승인 대기’는 빈 목록이 됩니다. 주소는 왕십리로 20으로 바뀝니다. 실제 실행 목록 끝에는 새길 10에서 왕십리로 20으로 바꾼 내역이 붙습니다.

완료 시점도 `original_end`에 저장해 둡니다. 실제 모델이 승인할 호출을 내지 않았다면 재개 생략 안내가 표시됩니다.

In [ ]:
original_result = approve_delivery(original_agent, original_config)
if original_result is not None:
    show_delivery(original_agent, original_config, original_result, original_shop)
original_end = original_agent.get_state(original_config)

### 미들웨어가 오류를 받은 뒤 자동 재시도

「자동 재시도」 셀(`auto-retry`)은 같은 요청으로 비교합니다. 새 장난감 데이터의 첫 조회도 시간 초과입니다. `max_retries=1`이라서 미들웨어가 `call-1`을 한 번 더 실행합니다. 성공한 응답 하나만 메시지에 남습니다. 모델은 조회를 한 번 요청했고 함수는 두 번 실행됐습니다. 위 예시의 호출 ID, `status`, 실행 목록과 비교하세요.

이 비교는 둘 다 첫 변경 승인 직전까지입니다. 여기서 출력하는 승인 문구와 배송지는 아직 변경 전임을 보여 줍니다. 이후 재생은 앞에서 승인까지 마친 원래 대화로 돌아갑니다.

In [ ]:
auto_shop = make_toy_shop()
auto_agent = make_delivery_agent(auto_shop, tool_retries=1)
auto_config = thread_config("delivery-auto-retry")
auto_pause = auto_agent.invoke({"messages": [customer]}, auto_config)
show_delivery(auto_agent, auto_config, auto_pause, auto_shop)

## 2. 조회 완료 시점을 찾아 재생하기

대화 하나에는 `thread_id` 하나가 있고, 그 아래 여러 체크포인트가 쌓입니다. `checkpoint_id`는 저장 시점 하나를 고릅니다. 그 시점의 `values["messages"]`에는 그때까지의 대화와 도구 응답이 있습니다. `next`는 다음에 실행할 노드이고 빈 튜플 `()`이면 할 일이 없습니다. 주문 딕셔너리와 실제 실행 목록은 이 상태 바깥에 있습니다.

「기록과 재생할 시점」 셀(`find-checkpoint`)의 `get_state_history`는 최신 체크포인트부터 읽습니다. 각 줄의 `step`은 저장 단계 번호입니다. `id`와 `parent`는 현재 체크포인트와 바로 앞 체크포인트 ID의 끝 여덟 글자입니다. `messages`는 저장된 메시지 수입니다. `next`에 `model`이나 `tools`가 보이면 모델이나 도구를 실행할 차례입니다. `HumanInTheLoopMiddleware.after_model`은 모델 응답 뒤에 미들웨어가 거치는 단계입니다.

조회 성공 메시지에서 끝나는 체크포인트 ID를 찾습니다. 그 ID에 해당하는 설정을 기록에서 꺼내 `get_state`에 넘기면 정확히 그 시점이 열립니다. 저장된 설정에는 `thread_id`와 `checkpoint_id`가 함께 들어 있습니다. 이어서 전체 ID, 당시의 메시지, 다음 노드, 현재 배송지를 출력합니다. 가짜 모델에서는 다음 노드가 `model`이고 당시 변경 요청은 아직 없습니다. 현재 주소는 이미 왕십리로 20입니다. 선택할 조회 성공 기록이 없는 경우에는 안내를 띄웁니다.

In [ ]:
def checkpoint_id(config):
    return config["configurable"]["checkpoint_id"] if config is not None else None


def short_id(config):
    saved_id = checkpoint_id(config)
    return saved_id[-8:] if saved_id else "—"


history = list(original_agent.get_state_history(original_config))
print("대화:", original_config["configurable"]["thread_id"])
lookup_checkpoint_id = None
for saved in history:
    messages = saved.values.get("messages", [])
    print(f"step={saved.metadata['step']} | id={short_id(saved.config)} | "
          f"parent={short_id(saved.parent_config)} | messages={len(messages)} | next={saved.next}")
    if saved.next == ("model",) and messages:
        last_message = messages[-1]
        if (lookup_checkpoint_id is None and isinstance(last_message, ToolMessage)
                and last_message.name == "check_delivery_window" and last_message.status == "success"):
            lookup_checkpoint_id = checkpoint_id(saved.config)

lookup_config = None
if lookup_checkpoint_id is None:
    print("성공한 배송 조회 직후의 체크포인트를 찾지 못했습니다.")
else:
    lookup_config = next(
        saved.config for saved in history if checkpoint_id(saved.config) == lookup_checkpoint_id
    )
    lookup_checkpoint = original_agent.get_state(lookup_config)
    print("고른 checkpoint_id:", lookup_checkpoint_id)
    show(lookup_checkpoint.values["messages"])
    print("저장된 다음 노드:", lookup_checkpoint.next)
    print("지금의 배송지:", original_shop["orders"]["O001"]["address"])

### 같은 체크포인트에서 재생

「조회 뒤 재생」 셀(`replay`)은 `invoke(None, lookup_config)`로 이어 갑니다. `None`은 저장된 입력으로 이어 가라는 뜻입니다. 모델부터 다시 실행해 메시지에 있는 조회 결과를 읽습니다. 승인이 다시 뜨면 새 중단 ID로 재개합니다.

첫 출력에서는 변경 승인이 다시 대기 중이고, 주소와 실행 목록은 앞선 완료 시점과 같습니다. 승인 후에는 왕십리로 20에서 같은 주소로 다시 쓰는 변경 한 건이 목록에 추가됩니다. 실제 실행 목록에는 앞서 했던 조회 두 건과 변경 두 건이 보입니다. 이 장난감 도구는 같은 주소를 쓰는 작업도 기록합니다. 재생 전에 주문이 이미 바뀌어 있다는 것을 이 기록에서 확인할 수 있습니다.

재생이 끝난 지점은 `replay_end`에 저장합니다. 조회 체크포인트가 없으면 재생 셀은 생략 안내만 표시합니다.

In [ ]:
replay_end = None
if lookup_config is None:
    print("조회 체크포인트가 없어 재생을 건너뜁니다.")
else:
    replay_pause = original_agent.invoke(None, lookup_config)
    print("재생 후, 승인 전")
    show_delivery(original_agent, original_config, replay_pause, original_shop)
    replay_result = approve_delivery(original_agent, original_config)
    if replay_result is not None:
        print("재생 후, 승인 완료")
        show_delivery(original_agent, original_config, replay_result, original_shop)
        replay_end = original_agent.get_state(original_config)

### 그 시점에서 다른 배송지로 분기

「새 배송지로 분기」 셀(`fork-address`)은 `update_state`로 과거 체크포인트에서 새 체크포인트를 만듭니다. 기존 고객 메시지와 **같은 메시지 ID**로 새 문장을 넘겨 그 한 항목을 교체합니다. 조회는 출고 상태만 확인했으므로 새 주소에서도 그 결과를 쓸 수 있습니다.

`as_node="tools"`는 이 수정을 조회 도구 단계가 남긴 상태처럼 저장합니다. 그래서 다음 노드가 `model`입니다. `update_state`가 준 새 설정으로 `invoke(None, fork_config)`를 호출하면 모델은 바뀐 주소를 읽습니다. 이때 출력하는 다음 노드와 체크포인트 끝자리는 새 출발점을 가리킵니다.

승인 전에는 성수이로 30을 묻는 문구와 왕십리로 20인 현재 배송지가 함께 보입니다. 승인하면 실제 실행 목록에 왕십리로 20에서 성수이로 30으로 바꾼 기록이 추가됩니다. 원래 체크포인트 안의 고객 요청은 다음 셀에서 다시 확인합니다. 출발점이 없거나 변경할 고객 메시지가 없으면 생략 안내를 표시합니다.

In [ ]:
fork_config = None
fork_end = None
if lookup_config is None or not any(
    message.id == customer.id for message in lookup_checkpoint.values["messages"]
):
    print("조회 체크포인트나 교체할 고객 메시지가 없어 분기를 건너뜁니다.")
else:
    fork_address = "서울시 성동구 성수이로 30"
    revised_customer = HumanMessage(
        id=customer.id,
        content=f"주문 O001의 배송지를 바꿔 주세요. 새 배송지: {fork_address}",
    )
    fork_config = original_agent.update_state(
        lookup_config, {"messages": [revised_customer]}, as_node="tools",
    )
    print("분기 시작 id:", short_id(fork_config), "| next:", original_agent.get_state(fork_config).next)

    fork_pause = original_agent.invoke(None, fork_config)
    print("분기 후, 승인 전")
    show_delivery(original_agent, original_config, fork_pause, original_shop)
    fork_result = approve_delivery(original_agent, original_config)
    if fork_result is not None:
        print("분기 후, 승인 완료")
        show_delivery(original_agent, original_config, fork_result, original_shop)
        fork_end = original_agent.get_state(original_config)

### 한 대화 안에서 갈라진 경로 읽기

「분기 기록」 셀(`branch-history`)은 같은 `thread_id`의 기록을 다시 읽습니다. 완료 지점마다 `parent_config`를 거슬러 올라간 뒤, 조회 체크포인트부터 완료까지 화살표로 표시합니다. 맨 앞 ID는 공통입니다. 그 다음 ID가 갈라지는 지점과 ‘분기 시작 id’가 어느 경로에 들어가는지 찾아보세요.

각 경로 밑에는 그 완료 기록에 저장된 고객 요청이 있습니다. 원래 실행과 재생에는 왕십리로 20, 새 분기에는 성수이로 30이 남습니다. 마지막에 출력하는 실제 주소는 공유하는 주문의 현재 값입니다. 완료하지 않은 경로가 있으면 분기 비교를 생략했다는 안내를 표시합니다.

`InMemorySaver`의 기록은 커널을 재시작하면 사라집니다. 이 실습의 주문 딕셔너리와 실행 목록도 함께 사라집니다.

In [ ]:
endpoints = (("원래 실행", original_end), ("재생", replay_end), ("새 주소 분기", fork_end))
if original_result is None or any(saved is None or saved.next for _, saved in endpoints):
    print("아직 세 경로가 모두 완료되지 않아 분기 비교를 건너뜁니다.")
else:
    branched_history = {
        checkpoint_id(saved.config): saved
        for saved in original_agent.get_state_history(original_config)
    }
    print("세 경로의 thread_id:", original_config["configurable"]["thread_id"])
    for label, endpoint in endpoints:
        path = [short_id(endpoint.config)]
        cursor = endpoint
        while checkpoint_id(cursor.config) != lookup_checkpoint_id:
            cursor = branched_history[checkpoint_id(cursor.parent_config)]
            path.append(short_id(cursor.config))
        print(label, " → ".join(reversed(path)))
        show([message for message in endpoint.values["messages"] if message.id == customer.id])
    print("장난감 주문의 실제 주소:", original_shop["orders"]["O001"]["address"])

### shopagent에서 이어 읽기

| 본 개념 | 프로젝트 위치와 역할 |
|---|---|
| 모델 연결 | [shop/agents.py](../shop/agents.py)의 `LocalLLM`과 [shop/servers.py](../shop/servers.py)의 `CONFIGS`. Gemma 연결과 `solar-pro4` 설정을 찾을 수 있습니다. |
| 배송지 변경과 실행 기록 | [shop/tools.py](../shop/tools.py)의 `Shop.change_shipping_address`와 `Shop.record_call`. 주문 주소를 바꾸는 코드와 도구 실행을 기록하는 코드입니다. |
| 도구 실행 감싸기와 승인 | [shop/graph_v2.py](../shop/graph_v2.py)의 `TurnMiddleware.wrap_tool_call`과 `build_agent`. 도구 실행을 감싸고 중요한 변경을 승인 직전에 멈춥니다. |
| 체크포인터, 대화, 재개 | 같은 파일의 `ScenarioGraph.__init__`, `build_graph`, `ScenarioGraph.resume_turn`. 바깥 그래프에 저장소와 대화 이름표를 붙이고 `Command`로 중단을 재개합니다. |

## 직접 바꿔 볼 것

`USE = "fake"`로 하세요. 한 항목씩 바꾼 뒤 적힌 셀을 실행하고, 다음 실험 전에는 원래 값을 되돌리세요. 4~6번은 각각 「모델 재시도」 셀(`model-retry`)부터 「기록과 재생할 시점」 셀(`find-checkpoint`)까지 원래 코드로 다시 실행하고 시작하세요.

1. **첫 시도 성공.** 「모델 재시도」 셀(`model-retry`)의 `make_toy_shop()`을 `make_toy_shop(failures=0)`으로 바꿔 실행하세요. 조회 한 건과 성공 메시지 하나로 변경 승인까지 갑니다. 실제 주소는 아직 새길 10입니다.
2. **두 번 연속 실패.** 같은 셀의 값을 `failures=2`로 바꿔 실행하세요. 조회 오류 메시지가 두 개 보입니다. 모델은 더 요청하지 않고 변경하지 않았다고 답합니다. 승인 대기도 비어 있습니다.
3. **자동 재시도 끄기.** 「자동 재시도」 셀(`auto-retry`)의 `tool_retries=1`을 0으로 바꿔 그 셀을 실행하세요. 함수 실행은 여전히 두 번입니다. 이번에는 오류 메시지가 대화에 남고 모델이 새 호출 ID를 보냅니다.
4. **완료 지점에서 재생.** 「조회 뒤 재생」 셀(`replay`)의 첫 `invoke`를 `original_agent.invoke(None, original_end.config)`로 바꿔 그 셀을 실행하세요. 완료 체크포인트에는 남은 일이 없어 변경 승인이 뜨지 않고 실행 목록도 늘지 않습니다. 이어서 그 셀의 원래 코드를 되돌리세요.
5. **조회 직전으로 더 돌아가기.** 같은 셀의 첫 `invoke`에 `lookup_config` 대신 `lookup_checkpoint.parent_config`를 넣어 실행하세요. 가짜 모델에서 이곳의 다음 작업은 성공했던 두 번째 조회입니다. 장난감 배송 서비스의 오류 횟수는 이미 소진됐으므로 조회 성공이 한 번 더 기록됩니다. 변경 승인 후에는 변경 작업도 한 건 늘어납니다.
6. **세 번째 주소 써 보기.** 원래 코드로 「조회 뒤 재생」까지 실행한 뒤 「새 배송지로 분기」 셀(`fork-address`)의 `fork_address`를 `"서울시 성동구 연무장길 40"`으로 바꾸고 「분기 기록」까지 실행하세요. 새 경로에만 연무장길 40 요청이 보이며 실제 주소도 그곳으로 바뀝니다. 원래 경로와 조회 체크포인트는 남아 있습니다.